# Fertilizer RAS — toy 5-country example

This notebook is a *thin* driver: all the logic lives in `src/`. We just

1. build a tiny 5-country dataset (Russia / China / USA / India / Brazil),
2. call `FertilizerRAS(...).run()`,
3. show the main outputs and a couple of plots.

See `docs/methodology.md` for the equations implemented in each phase.

In [2]:
import sys, pathlib

sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd

from src.model import FertilizerRAS
from src.postprocessing import sanity_checks
from src.utils import plot_sankey, plot_heatmap, plot_country_bars

pd.set_option('display.float_format', '{:,.1f}'.format)

## 1. Toy input data (post-shock Production and Demand, Historical trade flow)

Units: thousands of tonnes (kt). Assumption: Russia cut 60 %, China cut 30 %.

In [3]:
countries = ['Russia', 'China', 'USA', 'India', 'Brazil']

P = pd.Series([3_000, 6_000, 5_500, 2_500, 1_000], index=countries, name='Production_P')
C = pd.Series([1_200, 6_500, 4_000, 5_000, 3_500], index=countries, name='Demand_C')

T0_data = {
    'Russia': [   0,   500,   300, 1_200,   800],
    'China':  [ 100,     0,   200, 1_500,   400],
    'USA':    [ 200,   300,     0,   400,   800],
    'India':  [   0,    50,     0,     0,     0],
    'Brazil': [   0,     0,    50,     0,     0],
}
T0 = pd.DataFrame(T0_data, index=countries, columns=countries).T

display(P.to_frame())
display(C.to_frame())
display(T0)

,Production_P
Russia,3000
China,6000
USA,5500
India,2500
Brazil,1000


,Demand_C
Russia,1200
China,6500
USA,4000
India,5000
Brazil,3500


,Russia,China,USA,India,Brazil
Russia,0,500,300,1200,800
China,100,0,200,1500,400
USA,200,300,0,400,800
India,0,50,0,0,0
Brazil,0,0,50,0,0


## 1b. Input-data visualizations

Before running the RAS, let's visualize the three inputs:

- a **choropleth** of post-shock production ($P$),
- a **choropleth** of post-shock demand ($C$),
- a **chord diagram** of the historical trade matrix ($T_0$),
- a **geographic network** of the same historical trade matrix on a world map.

After RAS, §3b repeats the same views for the final availability ($F$) and trade matrix ($X$).

In [4]:
# Force a re-execution of src.utils in case the kernel cached an older copy
# (e.g. when it was imported in an earlier cell before these plotters existed).
import importlib
import src.utils
importlib.reload(src.utils)

from src.utils import (
    plot_production_map,
    plot_demand_map,
    plot_chord_diagram,
    plot_trade_network_map,
)

plot_production_map(P, title='Toy example — post-shock production (P)').show()
plot_demand_map(C, title='Toy example — post-shock demand (C)').show()
plot_chord_diagram(T0, title='Toy example — historical trade matrix (T0)').show()
plot_trade_network_map(T0, title='Toy example — historical trade network (T0)').show()

## 2. Run the 4-phase pipeline

In [5]:
# Force the kernel to re-read src.model after edits to the source
# (same pattern as cell 5 uses for src.utils).
import importlib, src.model
importlib.reload(src.model)
from src.model import FertilizerRAS

# Collect iteration-by-iteration RAS trace (small toy only — avoid on huge matrices):
#   x_ras_iterations[k]  = X after iteration k+1 (post row- AND column-update)
#   r_updates[k]         = row multipliers r after iteration k+1 row step
#   c_updates[k]         = column multipliers c after iteration k+1 column step
x_ras_iterations: list[pd.DataFrame] = []
r_updates: list[pd.Series] = []
c_updates: list[pd.Series] = []

model = FertilizerRAS(P, C, T0)
result = model.run(
    verbose=True,
    x_history=x_ras_iterations,
    r_history=r_updates,
    c_history=c_updates,
)

print(f"\nRAS trace: {len(r_updates)} iterations recorded.\n")

# Table of all r-vectors (one column per iteration); show with 3 significant digits.
_fmt3 = '{:.3g}'.format
r_table = pd.concat(r_updates, axis=1) if r_updates else pd.DataFrame()
c_table = pd.concat(c_updates, axis=1) if c_updates else pd.DataFrame()
print("Row multipliers r per iteration (one column per iteration):")
display(r_table.style.format(_fmt3))
print("Column multipliers c per iteration:")
display(c_table.style.format(_fmt3))

# Per-iteration step-by-step view
for it, (rk, ck, Xk) in enumerate(zip(r_updates, c_updates, x_ras_iterations), start=1):
    print(f"\n=== Iteration {it} ===")
    print("After row update — r:")
    display(rk.to_frame(f"r_iter{it}").style.format(_fmt3))
    print("After column update — c:")
    display(ck.to_frame(f"c_iter{it}").style.format(_fmt3))
    print("Resulting trade matrix X = diag(r) · T · diag(c):")
    display(Xk.style.format(_fmt3))

# Full phase tables and balance checks are in the next cell.

RAS converged in 8 iterations (max_err=1.22e-07)

RAS trace: 8 iterations recorded.

Row multipliers r per iteration (one column per iteration):


,r_iter1,r_iter2,r_iter3,r_iter4,r_iter5,r_iter6,r_iter7,r_iter8
Brazil,0,0,0,0,0,0,0,0
China,0,0,0,0,0,0,0,0
India,0,0,0,0,0,0,0,0
Russia,0.643,0.634,0.633,0.633,0.633,0.633,0.633,0.633
USA,0.882,0.898,0.899,0.899,0.899,0.899,0.899,0.899


Column multipliers c per iteration:


,c_iter1,c_iter2,c_iter3,c_iter4,c_iter5,c_iter6,c_iter7,c_iter8
Brazil,1.23,1.22,1.22,1.22,1.22,1.22,1.22,1.22
China,0.512,0.512,0.512,0.512,0.512,0.512,0.512,0.512
India,1.33,1.34,1.34,1.34,1.34,1.34,1.34,1.34
Russia,0,0,0,0,0,0,0,0
USA,0,0,0,0,0,0,0,0



=== Iteration 1 ===
After row update — r:


...

,r_iter8
Brazil,0
China,0
India,0
Russia,0.633
USA,0.899


After column update — c:


,c_iter8
Brazil,1.22
China,0.512
India,1.34
Russia,0
USA,0


Resulting trade matrix X = diag(r) · T · diag(c):


,Brazil,China,India,Russia,USA
Brazil,0,0,0,0,0
China,0,0,0,0,0
India,0,0,0,0,0
Russia,620,162,1.02e+03,0,0
USA,880,138,482,0,0


## 2b. Inspect phases & mass balance

Prints every intermediate vector and `X`, and checks that `sum(S_hat) == sum(D_hat)` and that RAS row/column sums match the targets (plus `sanity_checks` from `src.postprocessing`).


In [6]:
# Detailed phase-by-phase checks (toy is small enough to print everything)
r = result

print("=== Phase 1 → 2: global totals (same names as _phase2 in model.py) ===")
S_total = r.S_star.sum()
D_total = r.D_star.sum()
print("S_total = sum(S_star) =", S_total)
print("D_total = sum(D_star) =", D_total)

print("\n=== Phase 2: scaled targets (RAS inputs; must match) ===")
print("sum(S_hat) =", r.S_hat.sum())
print("sum(D_hat) =", r.D_hat.sum())

print("\n=== Phase 3: RAS margins vs S_hat / D_hat ===")
row_err = float((r.X.sum(axis=1) - r.S_hat).abs().max())
col_err = float((r.X.sum(axis=0) - r.D_hat).abs().max())
print("max |row sums - S_hat| =", row_err)
print("max |col sums - D_hat| =", col_err)
print("rows match:", np.allclose(r.X.sum(axis=1), r.S_hat, rtol=0, atol=1e-5))
print("cols match:", np.allclose(r.X.sum(axis=0), r.D_hat, rtol=0, atol=1e-5))

print("\n=== Net balance B = P - C ===")
display((r.P - r.C).to_frame("B"))

print("\n--- Vectors & trade matrix ---")
display(r.P.to_frame("P"))
display(r.C.to_frame("C"))
display(r.K.to_frame("K"))
display(r.S_star.to_frame("S_star"))
display(r.D_star.to_frame("D_star"))
display(r.S_hat.to_frame("S_hat"))
display(r.D_hat.to_frame("D_hat"))
display(r.X)
display(r.F.to_frame("F"))

print("\n=== Automated mass-balance summary (same as postprocessing.sanity_checks) ===")
display(sanity_checks(r))
display(r.summary())


=== Phase 1 → 2: global totals (same names as _phase2 in model.py) ===
S_total = sum(S_star) = 3300.0
D_total = sum(D_star) = 5500.0

=== Phase 2: scaled targets (RAS inputs; must match) ===
sum(S_hat) = 3300.0
sum(D_hat) = 3300.0

=== Phase 3: RAS margins vs S_hat / D_hat ===
max |row sums - S_hat| = 1.2193436305096839e-07
max |col sums - D_hat| = 0.0
rows match: True
cols match: True

=== Net balance B = P - C ===


,B
Brazil,"-2,500.0"
China,-500.0
India,"-2,500.0"
Russia,"1,800.0"
USA,"1,500.0"



--- Vectors & trade matrix ---


,P
Brazil,"1,000.0"
China,"6,000.0"
India,"2,500.0"
Russia,"3,000.0"
USA,"5,500.0"


,C
Brazil,"3,500.0"
China,"6,500.0"
India,"5,000.0"
Russia,"1,200.0"
USA,"4,000.0"


...

,Brazil,China,India,Russia,USA
Brazil,0.0,0.0,0.0,0.0,0.0
China,0.0,0.0,0.0,0.0,0.0
India,0.0,0.0,0.0,0.0,0.0
Russia,619.9,162.0,"1,018.1",0.0,0.0
USA,880.1,138.0,481.9,0.0,0.0


,F
Brazil,"2,500.0"
China,"6,300.0"
India,"4,000.0"
Russia,"1,200.0"
USA,"4,000.0"



=== Automated mass-balance summary (same as postprocessing.sanity_checks) ===


,max_row_error,max_col_error,sum_X,sum_F,sum_K_plus_X,F_eq_K_plus_X,rows_ok,cols_ok
0,0.0,0.0,"3,300.0","18,000.0","18,000.0",True,True,True


,Production_P,Demand_C,Kept_K,Imports_received,F_final,Unmet_demand,Coverage_%
Brazil,"1,000.0","3,500.0","1,000.0","1,500.0","2,500.0","1,000.0",71.4
China,"6,000.0","6,500.0","6,000.0",300.0,"6,300.0",200.0,96.9
India,"2,500.0","5,000.0","2,500.0","1,500.0","4,000.0","1,000.0",80.0
Russia,"3,000.0","1,200.0","1,200.0",0.0,"1,200.0",0.0,100.0
USA,"5,500.0","4,000.0","4,000.0",0.0,"4,000.0",0.0,100.0


## 3. Plots

Sankey, heatmap and country-balance bars for the RAS result.

In [7]:
plot_sankey(result, title='Toy example — post-RAS trade flows').show()
plot_heatmap(result, title='Toy example — RAS trade matrix (X)').show()
plot_country_bars(result, title='Toy example — country balance').show()

## 3b. Post-RAS visualizations (mirror of §1b)

Same views as the input-data section, but for the **final** outcome:

- a **choropleth** of final fertilizer availability per country (`F` = domestic kept + imports received),
- a **chord diagram** of the post-RAS trade matrix (`X`),
- a **geographic network** of the same post-RAS flows on a world map.

In [8]:
from src.utils import (
    plot_availability_map,
    plot_chord_diagram,
    plot_trade_network_map,
)

plot_availability_map(result.F, title='Toy example — final availability per country (F)').show()
plot_chord_diagram(result.X, title='Toy example — post-RAS trade matrix (X)').show()
plot_trade_network_map(result.X, title='Toy example — post-RAS trade network (X)').show()